# P24 — Prédire la consommation (MPG) des véhicules américains (EPA fueleconomy.gov)

**Contexte métier.** L'EPA publie la consommation estimée de tous les véhicules vendus aux États-Unis depuis 1984 (données de l'étiquetage énergétique). Un assureur ou un courtier en flotte veut estimer la consommation combinée (MPG) d'un modèle à partir de ses caractéristiques techniques.

**Objectif de l'analyse exploratoire.** Identifier la cible (MPG combiné `comb08`) et les régresseurs (cylindrée, cylindres, année, classe, transmission), en excluant les véhicules électriques (MPGe).

**Données.** fueleconomy.gov (EPA/DOE), export « vehicles.csv » — 50 407 véhicules 1984-2027, 84 colonnes.

## 1. Téléchargement

Export officiel de fueleconomy.gov (zip d'environ 2 Mo).

In [ ]:
import os, urllib.request, zipfile

os.makedirs("data", exist_ok=True)
URL = "https://www.fueleconomy.gov/feg/epadata/vehicles.csv.zip"
if not os.path.exists("data/vehicles.csv.zip"):
    urllib.request.urlretrieve(URL, "data/vehicles.csv.zip")
with zipfile.ZipFile("data/vehicles.csv.zip") as z:
    z.extractall("data")
print("OK", os.path.getsize("data/vehicles.csv") // 1024, "Ko")

## 2. Chargement

84 colonnes : la plupart sont des sorties dérivées (city/highway MPG, CO2, coût) — sources de fuites pour la cible. On charge tout puis on garde les colonnes techniques.

In [ ]:
import pandas as pd

df = pd.read_csv("data/vehicles.csv", low_memory=False)
print(df.shape, "lignes x colonnes")
print("Années :", int(df["year"].min()), "-", int(df["year"].max()), " ; marques :", df["make"].nunique())
manq = df[["year", "make", "model", "cylinders", "displ", "drive", "VClass",
           "fuelType1", "comb08", "co2TailpipeGpm", "trany"]].isna().mean().mul(100).round(1)
print("% manquant (colonnes utiles) :")
print(manq[manq > 0].to_string())
print()
df["electrique"] = df["fuelType1"] == "Electricity"
print("Part d'électriques :", round(100 * df["electrique"].mean(), 1), "%")

## 3. Analyse

Distribution du MPG combiné sur les thermiques, corrélations, évolution par année et par classe ; baseline linéaire (cylindrée + cylindres).

In [ ]:
therm = df[(~df["electrique"]) & df["comb08"].notna() & df["displ"].notna()].copy()
print("Véhicules thermiques :", len(therm))
print(therm["comb08"].describe().round(1).to_string())
print()
print("Corrélations Spearman avec comb08 :")
print(therm[["cylinders", "displ", "year", "comb08"]].corr(method="spearman")["comb08"].round(2).to_string())
print()
print("MPG médian par année (extraits) :")
med = therm.groupby("year")["comb08"].median()
print(med.loc[[1984, 1990, 2000, 2010, 2015, 2020, 2025]].to_string())
print()
print("MPG médian par classe (thermiques, top 6) :")
print(therm.groupby("VClass")["comb08"].median().sort_values().head(6).to_string())
print()
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
d = therm.dropna(subset=["cylinders"]).sample(20000, random_state=42)
Xtr, Xte, ytr, yte = train_test_split(d[["displ", "cylinders"]], d["comb08"], test_size=0.25, random_state=42)
lr = LinearRegression().fit(Xtr, ytr)
import numpy as np
print("Baseline linéaire (displ, cylinders) : MAE =",
      round(float(np.abs(lr.predict(Xte) - yte).mean()), 1), "MPG ; R2 =",
      round(float(lr.score(Xte, yte)), 2))

## 4. Résultats observés (exécution du 16/10/2026)

- 50 407 véhicules x 84 colonnes ; millésimes 1984-2027, 146 marques
- part d'électriques : 3,2 % (1 608) ; thermiques analysés : 48 753
- MPG combiné (thermiques) : moyenne 20,7, médiane 20, 5 % : 13, 95 % : 30, max 59
- corrélations Spearman avec le MPG : cylindrée -0,82, cylindres -0,78, année +0,27
- MPG médian par millésime : 1984 : 19 ; 2010 : 20 ; 2015 : 22 ; 2020 : 23 ; 2025 : 22
- par classe : vans et pickups 13-15 MPG vs compactes autour de 30
- baseline linéaire (cylindrée + cylindres) : MAE 2,7 MPG, R2 0,52 — la montée en gamme (arbres, features catégorielles) doit dépasser nettement
- manquants : cylinders/displ 3,3 % (électriques et certaines années), drive 2,4 %

## 5. Variable cible et régresseurs

**Cible (régression) : `comb08`** — consommation urbaine+route combinée en MPG, sur les véhicules thermiques. Cible miroir possible : `co2TailpipeGpm` (g/mile), quasi équivalente.

**Régresseurs retenus :**
- `displ` (cylindrée, corr. -0,82) et `cylinders` (-0,78) : le cœur physique
- `VClass` (compact, SUV, pickup : 13 vs 30 MPG) et `drive` (2WD/4WD)
- `trany` (automatique/manuelle, nombre de rapports), `make`, `year`
- `fuelType1` (essence normale/premium, diesel)

**Pièges :**
- 81 colonnes dérivées (city08, highway08, co2, barrels, fuelCost...) sont des fonctions de la cible : les inclure est une fuite directe
- les électriques : comb08 devient un MPGe ; les exclure ou changer de cible (range, Wh/mile)
- valeurs sentinelles (-1) et manquants masqués (électriques sans cylindrée)
- les hybrides ont un comportement intermédiaire : créer un marqueur plutôt qu'une exclusion binaire
- le progrès technique fait monter le MPG dans le temps : un split aléatoire cache cette dérive ; tester sur les millésimes récents